## Imports

In [1]:
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib as mpl
import scipy.io as sio
import itertools

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


## Linear Autoencoders

In [3]:
class SCA_encoder(nn.Module):
    '''
    Input: a (T * N) matrix X, of N neurons recorded over T timesteps.
    Output: an affine mapping of X to K-dimensional space.
    '''
    def __init__(
        self,
        N: 100, # number of neurons
        K: 3 # dimensionality
    ):
        super().__init__()
        self.U = nn.Linear(N,K)

    def forward(self,x):
        encoded_x = self.U(x)
        return encoded_x

class SCA_decoder(nn.Module):
    '''
    Input: affine mapped X in K-dimensional space
    Output: X_hat, or the reconstructed X back in neural activity space
    '''
    def __init__(
        self,
        N: 100, # number of neurons
        K: 3 # dimensionality
    ):
        super().__init__()
        self.V = nn.Linear(K,N)

    def forward(self, encoded_x):
        normalized_V_weight = nn.functional.normalize(self.V.weight.data,dim=1)
        self.V.weight.data = normalized_V_weight
        x_hat = self.V(encoded_x)
        return x_hat
        

## Training Function

In [4]:
# Have not yet implemented sample-weighting
def train(X, encoder, decoder, optimizer, lambda_sparse = 0, lambda_orth = 0, sample_weighting=False):

    if sample_weighting:
        print("not yet!")
        return None

    X = X.to(device)

    # Calculate reconstruction loss
    encoded_X = encoder(X)
    decoded_X = decoder(encoded_X)
    reconstruction_loss = torch.norm(X - decoded_X)**2

    # Calculate sparsity penalty
    sparsity_loss = lambda_sparse * torch.norm(encoded_X)

    # Calculate orthogonality penalty
    V_weight = nn.functional.normalize(decoder.V.weight.data,dim=1)
    VTV = torch.matmul(V_weight.transpose(1,0),V_weight)
    I = torch.eye(VTV.size(dim=0),device=device)
    orth_loss = lambda_orth * torch.norm(VTV - I)**2

    # Sum to calculate loss
    loss = reconstruction_loss + sparsity_loss + orth_loss

    # Backpropagate

    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    return loss.item()


## Training Loop

In [14]:
def training_loop(X, encoder, decoder, lambda_sparse = 0, lambda_orth = 0, sample_weighting=False, epochs=100):

    lr = 1e-3
    min_lr = 5e-4
    plateau_len = 100
    factor = 0.5
    all_params = itertools.chain(encoder.parameters(), decoder.parameters())
    optimizer = torch.optim.Adam(all_params,lr=lr)
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=factor, patience=plateau_len, min_lr=min_lr)
    
    if sample_weighting:
        print("not yet!")
        return None
    losses = torch.zeros(epochs)
    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        loss_t = train(X, encoder, decoder, optimizer, lambda_sparse, lambda_orth, sample_weighting)
        print(f"loss: {loss_t:>7f}  [{t:>5d}/{epochs:>5d}]")
        losses[t] = loss_t
        if t % 100 == 0:
            current_lr = lr_scheduler.get_last_lr()[-1]
            print(f"learning rate: {current_lr:>7f}")
        lr_scheduler.step(loss_t)
    
    return losses

In [16]:
N = 10
K = 3
T = 15

encoder_ex = SCA_encoder(N=N,K=K).to(device)
decoder_ex = SCA_decoder(N=N,K=K).to(device)


data_ex = torch.randn(T,N)

epochs = 100000
training_loop(data_ex,encoder_ex,decoder_ex,lambda_sparse=0.1,lambda_orth=0.1,epochs=epochs)

Epoch 1
-------------------------------
loss: 291.541718  [    0/100000]
learning rate: 0.001000
Epoch 2
-------------------------------
loss: 290.306763  [    1/100000]
Epoch 3
-------------------------------
loss: 289.082306  [    2/100000]
Epoch 4
-------------------------------
loss: 287.868561  [    3/100000]
Epoch 5
-------------------------------
loss: 286.665710  [    4/100000]
Epoch 6
-------------------------------
loss: 285.473816  [    5/100000]
Epoch 7
-------------------------------
loss: 284.293060  [    6/100000]
Epoch 8
-------------------------------
loss: 283.123535  [    7/100000]
Epoch 9
-------------------------------
loss: 281.965332  [    8/100000]
Epoch 10
-------------------------------
loss: 280.818481  [    9/100000]
Epoch 11
-------------------------------
loss: 279.683044  [   10/100000]
Epoch 12
-------------------------------
loss: 278.559082  [   11/100000]
Epoch 13
-------------------------------
loss: 277.446503  [   12/100000]
Epoch 14
--------------

KeyboardInterrupt: 